### Extending P0 with Binary Constants

Following Python function is from the course notes:

In [ ]:
def runwasm(wasmfile):
  from IPython.display import display, Javascript

  with open(wasmfile, 'rb') as file:
    wasm_bytes = file.read()  # read the WASM file and convert its content to a comma-separated string of byte values
    wasm_byte_array_str = ','.join(str(byte) for byte in wasm_bytes)

  display(
    Javascript(f""" // Javascript code to compile and run the WASM module
    const wasmBinary = new Uint8Array([{wasm_byte_array_str}]); // convert back to binary representation
 
    const params = {{
        'P0lib': {{
            'write': i => element.append(i + ' '),
            'writeln': () => element.append(document.createElement('br')),
            'read': () => window.prompt()
        }}
    }};
 
    WebAssembly.compile(wasmBinary.buffer) // compile shareable code
        .then(module => WebAssembly.instantiate(module, params)) // create an instance with memory
        // .then(instance => instance.exports.program()); // run the main program; not needed if a start function is specified
    """)
  )

In [ ]:
def runpywasm(wasmfile):
  from pywasm.core import Machine, Runtime, FuncType, ValType

  # P0lib implementation in Python
  def write(_: Machine, args: list[int]) -> list[int]:
    print(args[0])
    return []

  def writeln(_: Machine, args: list[int]) -> list[int]:
    print()
    return []

  def read(_: Machine, args: list[int]) -> list[int]:
    return [int(input())]

  # Create runtime
  runtime = Runtime()
  runtime.imports = {
    'P0lib': {
      'write': runtime.allocate_func_host(FuncType([ValType.i32()], []), write),
      'writeln': runtime.allocate_func_host(FuncType([], []), writeln),
      'read': runtime.allocate_func_host(FuncType([], [ValType.i32()]), read),
    }
  }
  # Create and run instance
  instance = runtime.instance_from_file(wasmfile)

In [ ]:
def runwasmtime(wasmfile):
  from wasmtime import Engine, Store, Module, Linker, Func, FuncType, ValType

  # P0lib implementation in Python
  def write(i: int):
    print(i)

  def writeln():
    print()

  def read() -> int:
    return int(input())

  # Create engine, store, and module
  engine = Engine()
  store = Store(engine)
  module = Module(store.engine, open(wasmfile, 'rb').read())
  # Use Linker to create the P0lib library
  write_func = Func(store, FuncType([ValType.i32()], []), write)
  writeln_func = Func(store, FuncType([], []), writeln)
  read_func = Func(store, FuncType([], [ValType.i32()]), read)
  linker = Linker(engine)
  linker.define(store, 'P0lib', 'write', write_func)
  linker.define(store, 'P0lib', 'writeln', writeln_func)
  linker.define(store, 'P0lib', 'read', read_func)
  # Create and run an instance
  instance = linker.instantiate(store, module)

In [ ]:
import import_ipynb
from P0 import compileString

Modify the P0 compiler that comes with this assignment to allow for [hexadecimal constants](https://go.googlesource.com/proposal/+/master/design/19308-number-literals.md). For this, the grammar of `number` should be:

    number ::= digit {digit} | '0' 'b' binarydigit {binarydigit}
    digit ::= '0' | ... | '9'
    binarydigit ::= digit | 'A' | ... | 'F'

State which parts of the compiler need to be modified. A test case is below.

Only the **scanner** (`SC.ipynb`) needs to be modified, specifically the `number()` procedure. The parser, symbol table, and code generator are unaffected since they already handle `NUMBER` tokens with integer values.

**Grammar modification:**

```
number ::= '0' 'b' binarydigit {binarydigit} | digit {digit}
digit ::= '0' | ... | '9'
binarydigit ::= '0' | '1'
```

**Modification to `number()`:** When the first character is `'0'`, consume it and check if the next character is `'b'`. If so, parse subsequent binary digits (`'0'` or `'1'`) and accumulate the value in base 2. Otherwise, fall through to the existing decimal parsing loop:

```python
if ch == '0':
    getChar()
    if ch == 'b':
        getChar()
        if ch not in ('0', '1'): mark('binary digit expected')
        while ch == '0' or ch == '1':
            val = 2 * val + int(ch)
            getChar()
        if val >= 2**31: mark('number too large')
        return
while '0' <= ch <= '9':
    val = 10 * val + int(ch)
    getChar()
if val >= 2**31: mark('number too large')
```

In [ ]:
compileString(
  """
program hex
    const b = 0b101010
    if 0b010101 = b then write(0b00111) else write(0b11100)
""",
  'bin.wat',
)

In [ ]:
!wat2wasm bin.wat

In [ ]:
runwasm('bin.wasm')

In [ ]:
runpywasm('bin.wasm')

In [ ]:
runwasmtime('bin.wasm')